# Annadata — Fertilizer → ONNX

No TensorFlow. Accept dataset license on Kaggle if prompted.


In [ ]:
!pip install -q kaggle skl2onnx onnx
import json, os
from pathlib import Path
KAGGLE_USERNAME, KAGGLE_KEY = "aftaabkazi", "YOUR_KAGGLE_KEY_HERE"
home = Path.home() / ".kaggle"; home.mkdir(exist_ok=True)
(home / "kaggle.json").write_text(json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}))
os.chmod(home / "kaggle.json", 0o600)
!kaggle datasets download -d gdabhishek/fertilizer-prediction -p /content --unzip
print("done")


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

OUT = Path("/content/agrosight_exports"); OUT.mkdir(exist_ok=True)
csv = next(Path("/content").rglob("*ertilizer*.csv"))
df = pd.read_csv(csv)
colmap = {c.lower().strip(): c for c in df.columns}

def pick(*names):
    for n in names:
        if n.lower() in colmap:
            return colmap[n.lower()]
    raise KeyError(names)

temp, hum, moist = pick("Temparature", "Temperature"), pick("Humidity"), pick("Moisture")
n, p, k = pick("Nitrogen", "N"), pick("Phosphorous", "Phosphorus", "P"), pick("Potassium", "K")
soil, crop = pick("Soil Type", "Soil"), pick("Crop Type", "Crop")
target = pick("Fertilizer Name", "Fertilizer", "label")
soil_le, crop_le, y_le = LabelEncoder(), LabelEncoder(), LabelEncoder()
X = pd.DataFrame({
    "temperature": df[temp].astype(float), "humidity": df[hum].astype(float),
    "moisture": df[moist].astype(float), "N": df[n].astype(float),
    "P": df[p].astype(float), "K": df[k].astype(float),
    "soil": soil_le.fit_transform(df[soil].astype(str)),
    "crop": crop_le.fit_transform(df[crop].astype(str)),
}).astype(np.float32)
y = y_le.fit_transform(df[target].astype(str))
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=250, max_depth=14, random_state=42, n_jobs=-1)
rf.fit(Xtr, ytr)
acc = accuracy_score(yte, rf.predict(Xte))
print("accuracy", acc)
print(classification_report(yte, rf.predict(Xte), target_names=y_le.classes_))
onnx_model = convert_sklearn(rf, initial_types=[("float_input", FloatTensorType([None, 8]))], target_opset=12)
(OUT / "fertilizer.onnx").write_bytes(onnx_model.SerializeToString())
meta = {"features": list(X.columns), "classes": list(y_le.classes_),
        "soil_classes": list(soil_le.classes_), "crop_classes": list(crop_le.classes_),
        "accuracy": round(float(acc)*100, 2)}
(OUT / "fertilizer_meta.json").write_text(json.dumps(meta, indent=2))
print(meta)
from google.colab import files
files.download(str(OUT / "fertilizer.onnx"))
files.download(str(OUT / "fertilizer_meta.json"))
